In [1]:
import sys
sys.path.append('..')

In [32]:
# %pip install pandas numpy
# %pip install openpyxl

In [33]:
import pandas as pd
import numpy as np
import os
import json 

path="Sample_Data.xlsx"
# xl=pd.ExcelFile(path)
# xl.sheet_names
df=pd.read_excel(path)
df.head()

,Week,Product,Sales_Units,Cost_of_Goods_Sold_COGS,Inventory_Level,Demand_Forecast,Price_Per_Unit,Gross_Profit,Operating_Profit,Revenue,Inventory_Turnover_Ratio,Gross_Profit_Margin,Revenue_Growth
0,1,Rice - Basmati,1153.0,21.84,2915,1242.0,29.29,8589.85,5362.48,33771.37,0.28,0.2544,0.0000
1,2,Rice - Basmati,1108.0,21.69,3307,1149.0,27.30,6215.88,3593.65,30248.40,0.38,0.2055,-0.1043
2,3,Rice - Basmati,1527.0,22.38,3280,1639.0,27.48,7787.70,2713.53,41961.96,0.46,0.1856,0.3872
3,4,Rice - Basmati,1299.0,21.41,1981,1093.0,26.56,6689.85,2756.27,34501.44,0.40,0.1939,-0.1778
4,5,Rice - Basmati,1346.0,21.69,3135,1424.0,26.27,6164.68,1804.61,35359.42,0.68,0.1743,0.0249


In [34]:
print("products", df.Product.nunique())
print(df.Product.value_counts().sort_index().to_string())
print("week range",df.Week.min(),df.Week.max(), "unique",df.Week.nunique())
# duplicates
print("duplicate rows",df.duplicated().sum())
print("dup product-week",df.duplicated(["Week","Product"]).sum())
# numeric summary and suspicious values
print(df.describe().T.to_string())
for c in df.select_dtypes("number").columns:
    print(c, "neg", (df[c]<0).sum(), "zero",(df[c]==0).sum())


products 30
Product
Barley                   260
Canned Tomatoes          260
Canned Tuna              260
Chickpeas                261
Cocoa Powder             261
Coffee - Arabica         261
Coffee - Robusta         262
Corn                     260
Flour - All Purpose      263
Lentils - Green          261
Lentils - Red            262
Milk Powder - Skimmed    260
Milk Powder - Whole      260
Oats                     262
Palm Oil                 261
Pasta - Spaghetti        262
Rice - Basmati           263
Rice - White             262
Salt - Iodized           263
Soybean Oil              262
Spices - Black Pepper    263
Spices - Turmeric        262
Sugar - Raw Brown        260
Sugar - White Refined    262
Sunflower Oil            260
Tea - Black Ceylon       260
Tea - Green              261
Wheat - Hard Red         262
Wheat - Soft             261
Yeast - Dry              262
week range 1 260 unique 260
duplicate rows 39
dup product-week 39
                           count          me

c:\Users\msii\Desktop\work\interview\armani\armani_prj\venv_armani\Lib\site-packages\pandas\core\nanops.py:1028: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


In [35]:
dups=df[df.duplicated(["Week","Product"],keep=False)].sort_values(["Product","Week"])
# dups[["Week","Product","Sales_Units","Demand_Forecast","Price_Per_Unit","Revenue"]].to_string(index=False)[:6000]
dups[["Week","Product","Sales_Units","Demand_Forecast","Price_Per_Unit","Revenue"]]


,Week,Product,Sales_Units,Demand_Forecast,Price_Per_Unit,Revenue
3636,241,Chickpeas,986.0,1072.0,18.26,18004.36
3637,241,Chickpeas,986.0,1072.0,18.26,18004.36
5416,196,Cocoa Powder,1106.0,1418.0,43.11,47679.66
5417,196,Cocoa Powder,1106.0,1418.0,43.11,47679.66
4899,202,Coffee - Arabica,1937.0,1804.0,31.37,60763.69
...,...,...,...,...,...,...
1033,246,Wheat - Soft,0.0,2227.0,42.83,0.00
7234,183,Yeast - Dry,0.0,4045.0,37.65,0.00
7235,183,Yeast - Dry,0.0,4045.0,37.65,0.00
7304,252,Yeast - Dry,1500.0,2945.0,35.72,53580.00


In [36]:
# %pip install python-dotenv
%pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [37]:
from __future__ import annotations

import json
import logging
import os
import uuid
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
# from sqlalchemy import create_engine, text
from sqlalchemy import create_engine as sqlalchemy_create_engine, text
from sqlalchemy.engine import Engine


In [38]:
# ============================================================
# CONFIGURATION
# ============================================================
from pathlib import Path
from dotenv import load_dotenv

BASE_DIR = Path.cwd()

# BASE_DIR = Path(__file__).resolve().parent
load_dotenv(BASE_DIR / ".env")
# load_dotenv(".env")

True

In [39]:




load_dotenv(BASE_DIR / ".env")
# load_dotenv(".env")

USER_NAME = os.getenv("USER_NAME")
PASSWORD = os.getenv("PASSWORD")
HOST = os.getenv("HOST")
PORT = os.getenv("PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = rf"postgresql+psycopg2://{USER_NAME}:{PASSWORD}@{HOST}:{PORT}/{DB_NAME}"

EXCEL_FILE = BASE_DIR / os.getenv(
# EXCEL_FILE = os.getenv(
    "EXCEL_FILE",
    "Sample_Data.xlsx",
)

if not DATABASE_URL:
    raise RuntimeError(
        "DATABASE_URL is not configured in .env"
    )


In [40]:
# ============================================================
# LOGGING
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
)

logger = logging.getLogger(__name__)

In [41]:
# ============================================================
# SOURCE SCHEMA
# ============================================================

EXPECTED_COLUMNS = [
    "week",
    "product",
    "sales_units",
    "cost_of_goods_sold_cogs",
    "inventory_level",
    "demand_forecast",
    "price_per_unit",
    "gross_profit",
    "operating_profit",
    "revenue",
    "inventory_turnover_ratio",
    "gross_profit_margin",
    "revenue_growth",
]


NUMERIC_COLUMNS = [
    "week",
    "sales_units",
    "cost_of_goods_sold_cogs",
    "inventory_level",
    "demand_forecast",
    "price_per_unit",
    "gross_profit",
    "operating_profit",
    "revenue",
    "inventory_turnover_ratio",
    "gross_profit_margin",
    "revenue_growth",
]


In [42]:
# These are the numeric columns that can be median-imputed
# in the warehouse layer.
#
# Sales_Units is deliberately excluded because rows with
# missing Sales_Units are rejected.
IMPUTABLE_COLUMNS = [
    "cost_of_goods_sold_cogs",
    "inventory_level",
    "demand_forecast",
    "price_per_unit",
    "gross_profit",
    "operating_profit",
    "revenue",
    "inventory_turnover_ratio",
    "gross_profit_margin",
    "revenue_growth",
]


# ============================================================
# DATABASE ENGINE
# ============================================================

def create_database_engine() -> Engine:
    """Create PostgreSQL SQLAlchemy engine."""

    return sqlalchemy_create_engine(
        DATABASE_URL,
        pool_pre_ping=True,
        future=True,
    )


In [43]:
def normalize_column_name(column: str) -> str:
    """
    Normalize a column name.

    This is only applied to the in-memory dataframe so that
    we can validate and process the source.

    The RAW values themselves are not modified.
    """

    column = str(column).strip()

    replacements = {
        " ": "_",
        "-": "_",
        "/": "_",
        "\\": "_",
        "(": "",
        ")": "",
    }

    for old, new in replacements.items():
        column = column.replace(old, new)

    return column.lower()


In [44]:
def validate_columns(df: pd.DataFrame) -> None:
    """
    Validate source column names.

    No data rows are changed here.
    """

    actual_columns = [
        normalize_column_name(column)
        for column in df.columns
    ]

    expected = set(EXPECTED_COLUMNS)
    actual = set(actual_columns)

    missing = expected - actual
    unexpected = actual - expected

    if missing:
        raise ValueError(
            "Missing required columns: "
            + ", ".join(sorted(missing))
        )

    if unexpected:
        raise ValueError(
            "Unexpected columns found: "
            + ", ".join(sorted(unexpected))
        )


In [45]:

# ============================================================
# DATABASE SCHEMA
# ============================================================

from sql_scripts import *

In [46]:
def create_database_objects(
    engine: Engine,
) -> None:
    """Create schemas and tables."""

    logger.info(
        "Creating PostgreSQL database objects."
    )

    with engine.begin() as connection:

        for statement in CREATE_SCHEMA_SQL.split(";"):

            statement = statement.strip()

            if statement:
                connection.execute(
                    text(statement)
                )

        connection.execute(
            text(CREATE_RUN_TABLE_SQL)
        )

        connection.execute(
            text(CREATE_RAW_TABLE_SQL)
        )

        connection.execute(
            text(CREATE_STAGING_TABLE_SQL)
        )

        for statement in CREATE_WAREHOUSE_TABLES_SQL.split(";"):

            statement = statement.strip()

            if statement:
                connection.execute(
                    text(statement)
                )


In [47]:
def create_run(
    engine: Engine,
    source_file: str,
) -> str:
    """
    Create a unique ETL run.

    Returns:
        run_id
    """

    run_id = str(uuid.uuid4())

    run_datetime = datetime.now(timezone.utc)

    sql = """
    INSERT INTO etl.etl_run (
        run_id,
        run_datetime,
        source_file,
        status
    )
    VALUES (
        :run_id,
        :run_datetime,
        :source_file,
        'RUNNING'
    );
    """

    with engine.begin() as connection:

        connection.execute(
            text(sql),
            {
                "run_id": run_id,
                "run_datetime": run_datetime,
                "source_file": source_file,
            },
        )

    logger.info(
        "Started ETL run: %s",
        run_id,
    )

    return run_id

In [48]:
def update_run(
    engine: Engine,
    run_id: str,
    status: str,
    **kwargs,
) -> None:
    """Update ETL run metadata."""

    allowed = {
        "raw_row_count",
        "staging_row_count",
        "warehouse_row_count",
        "duplicate_count",
        "rejected_count",
        "error_message",
    }

    updates = {
        key: value
        for key, value in kwargs.items()
        if key in allowed
    }

    updates["status"] = status

    assignments = ", ".join(
        f"{key} = :{key}"
        for key in updates
    )

    sql = f"""
    UPDATE etl.etl_run
    SET {assignments}
    WHERE run_id = :run_id;
    """

    updates["run_id"] = run_id

    with engine.begin() as connection:

        connection.execute(
            text(sql),
            updates,
        )


In [49]:
def read_source(
    path: Path,
) -> pd.DataFrame:
    """
    Read Excel.

    Values are not cleaned here.
    """

    if not path.exists():
        raise FileNotFoundError(
            f"Source file does not exist: {path}"
        )

    logger.info(
        "Reading source file: %s",
        path,
    )

    df = pd.read_excel(
        path,
        engine="openpyxl",
        dtype=object,
    )

    logger.info(
        "Source contains %d rows.",
        len(df),
    )

    return df

In [50]:
def prepare_raw_dataframe(
    df: pd.DataFrame,
    run_id: str,
) -> pd.DataFrame:
    """
    Prepare raw dataframe.

    Important:
        dtype=object was used when reading Excel.

    We convert values to their string representation for the
    database because RAW is intended to preserve what the
    source supplied without imposing the warehouse schema.

    No analytical transformations are performed.
    """

    raw = df.copy()

    # Normalize ONLY the column names so the database can have
    # stable identifiers.
    raw.columns = [
        normalize_column_name(column)
        for column in raw.columns
    ]

    # Source row number allows us to trace a row back to Excel.
    raw.insert(
        0,
        "source_row_number",
        range(2, len(raw) + 2),
    )

    raw.insert(
        0,
        "run_id",
        run_id,
    )

    # Convert source values to strings while preserving NULL.
    for column in EXPECTED_COLUMNS:

        raw[column] = raw[column].apply(
            lambda value: (
                None
                if pd.isna(value)
                else str(value)
            )
        )

    return raw

In [51]:
def load_raw(
    engine: Engine,
    raw_df: pd.DataFrame,
) -> None:
    """Append raw source data."""

    columns = [
        "run_id",
        "source_row_number",
        *EXPECTED_COLUMNS,
    ]

    raw_df = raw_df[columns]

    raw_df.to_sql(
        "source_sales",
        engine,
        schema="raw",
        if_exists="append",
        index=False,
        method="multi",
    )

    logger.info(
        "Loaded %d rows into raw.source_sales.",
        len(raw_df),
    )


In [52]:
# ============================================================
# STAGING
# ============================================================

def convert_staging_types(
    raw_df: pd.DataFrame,
    run_id: str,
) -> pd.DataFrame:
    """
    Convert source data into staging types.

    Invalid values become NULL.

    These conversion errors are later recorded as quality
    issues.
    """

    staging = raw_df.copy()

    # Product
    staging["product"] = (
        staging["product"]
        .astype("string")
        .str.strip()
    )

    # Numeric conversion
    for column in NUMERIC_COLUMNS:

        staging[column] = pd.to_numeric(
            staging[column],
            errors="coerce",
        )

    # Week
    staging["week"] = (
        staging["week"]
        .round()
        .astype("Int64")
    )

    # Sales units
    staging["sales_units"] = (
        staging["sales_units"]
        .round()
        .astype("Int64")
    )

    return staging


In [53]:
def find_type_conversion_issues(
    raw_df: pd.DataFrame,
    staging_df: pd.DataFrame,
    run_id: str,
) -> pd.DataFrame:
    """
    Identify values which failed numeric conversion.

    We compare RAW strings against the converted staging value.
    Empty strings are treated as missing rather than conversion
    errors.
    """

    issues = []

    for column in NUMERIC_COLUMNS:

        raw_values = raw_df[column]

        converted = staging_df[column]

        for index in raw_values.index:

            raw_value = raw_values.loc[index]
            converted_value = converted.loc[index]

            if raw_value is None:
                continue

            if str(raw_value).strip() == "":
                continue

            if pd.isna(converted_value):

                issues.append(
                    {
                        "run_id": run_id,
                        "source_row_number": int(
                            raw_df.loc[
                                index,
                                "source_row_number",
                            ]
                        ),
                        "product": raw_df.loc[
                            index,
                            "product",
                        ],
                        "week": None,
                        "column_name": column,
                        "issue_type": "TYPE_CONVERSION",
                        "issue_detail": (
                            f"Could not convert "
                            f"'{raw_value}' to numeric."
                        ),
                        "severity": "WARNING",
                    }
                )

    return pd.DataFrame(issues)

In [54]:
def remove_staging_duplicates(
    staging_df: pd.DataFrame,
    run_id: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Remove duplicates based on Product + Week.

    First occurrence is retained.

    Duplicate records are not lost completely:
    their information is written to the quality issue table.
    """

    duplicate_mask = staging_df.duplicated(
        subset=["product", "week"],
        keep="first",
    )

    duplicates = staging_df.loc[
        duplicate_mask
    ].copy()

    cleaned = staging_df.loc[
        ~duplicate_mask
    ].copy()

    issues = []

    for _, row in duplicates.iterrows():

        issues.append(
            {
                "run_id": run_id,
                "source_row_number": int(
                    row["source_row_number"]
                ),
                "product": (
                    None
                    if pd.isna(row["product"])
                    else str(row["product"])
                ),
                "week": (
                    None
                    if pd.isna(row["week"])
                    else int(row["week"])
                ),
                "column_name": None,
                "issue_type": "DUPLICATE_PRODUCT_WEEK",
                "issue_detail": (
                    "Duplicate Product + Week record "
                    "removed from staging."
                ),
                "severity": "WARNING",
            }
        )

    issue_df = pd.DataFrame(issues)

    logger.info(
        "Removed %d Product + Week duplicates.",
        len(duplicates),
    )

    return cleaned, issue_df

In [55]:
def load_staging(
    engine: Engine,
    staging_df: pd.DataFrame,
) -> None:
    """Load staging table."""

    columns = [
        "run_id",
        "source_row_number",
        *EXPECTED_COLUMNS,
    ]

    staging_df = staging_df[columns]

    staging_df.to_sql(
        "sales",
        engine,
        schema="staging",
        if_exists="append",
        index=False,
        method="multi",
    )

    logger.info(
        "Loaded %d rows into staging.sales.",
        len(staging_df),
    )


In [56]:
# ============================================================
# WAREHOUSE CLEANING
# ============================================================

def warehouse_clean(
    staging_df: pd.DataFrame,
    run_id: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Perform analytical/business cleaning.

    Rules
    -----
    1. Sales_Units missing:
           remove row

    2. Other numeric columns missing:
           median imputation by Product

    3. If product-level median itself is unavailable:
           global median

    4. Every modified row gets:
           row_modified = TRUE

    5. modified_columns contains names of modified fields.

    6. Original staging data remains untouched.
    """

    df = staging_df.copy()

    quality_issues = []

    # --------------------------------------------------------
    # Track modifications
    # --------------------------------------------------------

    df["row_modified"] = False
    df["modified_columns"] = None

    # --------------------------------------------------------
    # Sales_Units is mandatory
    # --------------------------------------------------------

    missing_sales = df["sales_units"].isna()

    rejected_sales = df.loc[
        missing_sales
    ].copy()

    for _, row in rejected_sales.iterrows():

        quality_issues.append(
            {
                "run_id": run_id,
                "source_row_number": int(
                    row["source_row_number"]
                ),
                "product": (
                    None
                    if pd.isna(row["product"])
                    else str(row["product"])
                ),
                "week": (
                    None
                    if pd.isna(row["week"])
                    else int(row["week"])
                ),
                "column_name": "sales_units",
                "issue_type": "MISSING_REQUIRED_VALUE",
                "issue_detail": (
                    "Row removed because Sales_Units "
                    "is missing."
                ),
                "severity": "ERROR",
            }
        )

    df = df.loc[
        ~missing_sales
    ].copy()

    # --------------------------------------------------------
    # Median imputation
    # --------------------------------------------------------

    for column in IMPUTABLE_COLUMNS:

        missing_mask = df[column].isna()

        if not missing_mask.any():
            continue

        # Product-level median
        product_median = (
            df.groupby("product")[column]
            .transform("median")
        )

        # Global fallback
        global_median = df[column].median()

        replacement = (
            product_median
            .fillna(global_median)
        )

        rows_to_modify = missing_mask & replacement.notna()

        if rows_to_modify.any():

            df.loc[
                rows_to_modify,
                column,
            ] = replacement.loc[
                rows_to_modify
            ]

            df.loc[
                rows_to_modify,
                "row_modified",
            ] = True

            for index in df.index[
                rows_to_modify
            ]:

                existing = (
                    df.loc[
                        index,
                        "modified_columns",
                    ]
                )

                if existing is None or pd.isna(existing):

                    modified_columns = []

                else:

                    modified_columns = list(
                        existing
                    )

                modified_columns.append(
                    column
                )

                df.at[
                    index,
                    "modified_columns",
                ] = modified_columns

    # --------------------------------------------------------
    # Business rule checks
    #
    # These DO NOT remove rows.
    # They create quality flags.
    # --------------------------------------------------------

    df["negative_value_flag"] = False

    non_negative_columns = [
        "sales_units",
        "cost_of_goods_sold_cogs",
        "inventory_level",
        "demand_forecast",
        "price_per_unit",
        "revenue",
    ]

    for column in non_negative_columns:

        mask = df[column] < 0

        if mask.any():

            df.loc[
                mask,
                "negative_value_flag",
            ] = True

            for index in df.index[mask]:

                quality_issues.append(
                    {
                        "run_id": run_id,
                        "source_row_number": int(
                            df.loc[
                                index,
                                "source_row_number",
                            ]
                        ),
                        "product": str(
                            df.loc[
                                index,
                                "product",
                            ]
                        ),
                        "week": (
                            int(
                                df.loc[
                                    index,
                                    "week",
                                ]
                            )
                            if pd.notna(
                                df.loc[
                                    index,
                                    "week",
                                ]
                            )
                            else None
                        ),
                        "column_name": column,
                        "issue_type": "NEGATIVE_VALUE",
                        "issue_detail": (
                            f"{column} is negative."
                        ),
                        "severity": "WARNING",
                    }
                )

    return (
        df,
        pd.DataFrame(quality_issues),
    )


In [57]:
# # ============================================================
# # FORMULA / CORRELATION CHECKS
# # ============================================================

# def calculate_formula_checks(
#     df: pd.DataFrame,
# ) -> pd.DataFrame:
#     """
#     Calculate derived values for validation.

#     IMPORTANT:
#         Source values are NOT overwritten.

#     We only create additional validation columns.
#     """

#     df = df.copy()

#     # --------------------------------------------------------
#     # Revenue
#     # --------------------------------------------------------

#     df["revenue_calculated"] = (
#         df["sales_units"]
#         * df["price_per_unit"]
#     )

#     df["revenue_difference"] = (
#         df["revenue"]
#         - df["revenue_calculated"]
#     )

#     df["revenue_check"] = np.isclose(
#         df["revenue"].astype(float),
#         df["revenue_calculated"].astype(float),
#         rtol=0.01,
#         atol=0.05,
#         equal_nan=True,
#     )

#     # --------------------------------------------------------
#     # Gross profit
#     #
#     # COGS is documented as unit acquisition cost.
#     # --------------------------------------------------------

#     total_cogs = (
#         df["sales_units"]
#         * df["cost_of_goods_sold_cogs"]
#     )

#     df["gross_profit_calculated"] = (
#         df["revenue_calculated"]
#         - total_cogs
#     )

#     df["gross_profit_difference"] = (
#         df["gross_profit"]
#         - df["gross_profit_calculated"]
#     )

#     df["gross_profit_check"] = np.isclose(
#         df["gross_profit"].astype(float),
#         df["gross_profit_calculated"].astype(float),
#         rtol=0.01,
#         atol=0.05,
#         equal_nan=True,
#     )

#     # --------------------------------------------------------
#     # Gross profit margin
#     # --------------------------------------------------------

#     df["gross_profit_margin_calculated"] = (
#         np.where(
#             df["revenue_calculated"] != 0,
#             df["gross_profit_calculated"]
#             / df["revenue_calculated"],
#             np.nan,
#         )
#     )

#     df["gross_profit_margin_difference"] = (
#         df["gross_profit_margin"]
#         - df["gross_profit_margin_calculated"]
#     )

#     df["gross_profit_margin_check"] = np.isclose(
#         df["gross_profit_margin"].astype(float),
#         df["gross_profit_margin_calculated"].astype(float),
#         rtol=0.01,
#         atol=0.001,
#         equal_nan=True,
#     )

#     return df

In [58]:
# ============================================================
# LOAD QUALITY ISSUES
# ============================================================

def load_quality_issues(
    engine: Engine,
    issue_frames: list[pd.DataFrame],
) -> None:
    """Append quality issues."""

    valid = [
        frame
        for frame in issue_frames
        if frame is not None
        and not frame.empty
    ]

    if not valid:
        return

    issues = pd.concat(
        valid,
        ignore_index=True,
    )

    issues.to_sql(
        "data_quality_issues",
        engine,
        schema="warehouse",
        if_exists="append",
        index=False,
        method="multi",
    )

    logger.info(
        "Recorded %d quality issues.",
        len(issues),
    )



In [59]:
# ============================================================
# DIMENSIONS
# ============================================================

def load_dimensions(
    engine: Engine,
    warehouse_df: pd.DataFrame,
) -> None:
    """Load product and week dimensions."""

    products = (
        warehouse_df[
            ["product"]
        ]
        .dropna()
        .drop_duplicates()
        .rename(
            columns={
                "product": "product_name"
            }
        )
    )

    weeks = (
        warehouse_df[
            ["week"]
        ]
        .dropna()
        .drop_duplicates()
        .rename(
            columns={
                "week": "week_number"
            }
        )
    )

    with engine.begin() as connection:

        for product in products[
            "product_name"
        ]:

            connection.execute(
                text(
                    """
                    INSERT INTO warehouse.dim_product (
                        product_name
                    )
                    VALUES (:product_name)
                    ON CONFLICT(product_name)
                    DO NOTHING;
                    """
                ),
                {
                    "product_name": str(product)
                },
            )

        for week in weeks[
            "week_number"
        ]:

            connection.execute(
                text(
                    """
                    INSERT INTO warehouse.dim_week (
                        week_number
                    )
                    VALUES (:week_number)
                    ON CONFLICT(week_number)
                    DO NOTHING;
                    """
                ),
                {
                    "week_number": int(week)
                },
            )



In [60]:

# ============================================================
# FACT TABLE
# ============================================================
def load_fact_table(
    engine: Engine,
    warehouse_df: pd.DataFrame,
    run_id: str,
) -> None:
    """
    Load the final warehouse fact table.

    No calculated columns are created.
    Source business values are preserved after
    warehouse cleaning.

    run_id is deliberately the final column.
    """

    records = []

    for _, row in warehouse_df.iterrows():

        records.append(
            {
                "source_row_number": int(
                    row["source_row_number"]
                ),

                "product": (
                    None
                    if pd.isna(row["product"])
                    else str(row["product"])
                ),

                "week": (
                    None
                    if pd.isna(row["week"])
                    else int(row["week"])
                ),

                "sales_units": int(
                    row["sales_units"]
                ),

                "unit_cogs": row[
                    "cost_of_goods_sold_cogs"
                ],

                "inventory_level": row[
                    "inventory_level"
                ],

                "demand_forecast": row[
                    "demand_forecast"
                ],

                "price_per_unit": row[
                    "price_per_unit"
                ],

                "gross_profit": row[
                    "gross_profit"
                ],

                "operating_profit": row[
                    "operating_profit"
                ],

                "revenue": row[
                    "revenue"
                ],

                "inventory_turnover_ratio": row[
                    "inventory_turnover_ratio"
                ],

                "gross_profit_margin": row[
                    "gross_profit_margin"
                ],

                "revenue_growth": row[
                    "revenue_growth"
                ],

                "row_modified": bool(
                    row["row_modified"]
                ),

                "modified_columns": (
                    json.dumps(
                        row["modified_columns"]
                    )
                    if isinstance(
                        row["modified_columns"],
                        list,
                    )
                    else None
                ),

                "run_id": run_id,
            }
        )

    fact_df = pd.DataFrame(records)

    # Temporary table
    fact_df.to_sql(
        "tmp_fact_sales",
        engine,
        schema="warehouse",
        if_exists="replace",
        index=False,
        method="multi",
    )

    sql = """
    INSERT INTO warehouse.fact_sales (

        source_row_number,

        product_key,
        week_key,

        sales_units,

        unit_cogs,
        inventory_level,
        demand_forecast,
        price_per_unit,

        gross_profit,
        operating_profit,
        revenue,

        inventory_turnover_ratio,
        gross_profit_margin,
        revenue_growth,

        row_modified,
        modified_columns,

        run_id
    )

    SELECT

        f.source_row_number,

        p.product_key,
        w.week_key,

        f.sales_units,

        f.unit_cogs,
        f.inventory_level,
        f.demand_forecast,
        f.price_per_unit,

        f.gross_profit,
        f.operating_profit,
        f.revenue,

        f.inventory_turnover_ratio,
        f.gross_profit_margin,
        f.revenue_growth,

        f.row_modified,
        CASE
            WHEN f.modified_columns IS NULL
            THEN NULL
            ELSE f.modified_columns::jsonb
        END,

        f.run_id

    FROM warehouse.tmp_fact_sales f

    JOIN warehouse.dim_product p
        ON p.product_name = f.product

    JOIN warehouse.dim_week w
        ON w.week_number = f.week

    ON CONFLICT (
        run_id,
        product_key,
        week_key
    )
    DO UPDATE SET

        source_row_number =
            EXCLUDED.source_row_number,

        sales_units =
            EXCLUDED.sales_units,

        unit_cogs =
            EXCLUDED.unit_cogs,

        inventory_level =
            EXCLUDED.inventory_level,

        demand_forecast =
            EXCLUDED.demand_forecast,

        price_per_unit =
            EXCLUDED.price_per_unit,

        gross_profit =
            EXCLUDED.gross_profit,

        operating_profit =
            EXCLUDED.operating_profit,

        revenue =
            EXCLUDED.revenue,

        inventory_turnover_ratio =
            EXCLUDED.inventory_turnover_ratio,

        gross_profit_margin =
            EXCLUDED.gross_profit_margin,

        revenue_growth =
            EXCLUDED.revenue_growth,

        row_modified =
            EXCLUDED.row_modified,

        modified_columns =
            EXCLUDED.modified_columns;

    """

    with engine.begin() as connection:

        connection.execute(text(sql))

        connection.execute(
            text(
                """
                DROP TABLE IF EXISTS
                warehouse.tmp_fact_sales;
                """
            )
        )

    logger.info(
        "Loaded %d warehouse fact rows.",
        len(fact_df),
    )

In [61]:

# ============================================================
# MAIN PIPELINE
# ============================================================

def run_pipeline() -> None:

    engine = create_database_engine()

    create_database_objects(engine)

    run_id = create_run(
        engine,
        EXCEL_FILE.name,
    )

    try:

        # ====================================================
        # 1. READ
        # ====================================================

        source_df = read_source(
            EXCEL_FILE
        )

        # ====================================================
        # 2. COLUMN VALIDATION
        # ====================================================

        validate_columns(
            source_df
        )

        # ====================================================
        # 3. RAW
        # ====================================================

        raw_df = prepare_raw_dataframe(
            source_df,
            run_id,
        )

        load_raw(
            engine,
            raw_df,
        )

        raw_count = len(raw_df)

        # ====================================================
        # 4. STAGING TYPE CONVERSION
        # ====================================================

        staging_df = convert_staging_types(
            raw_df,
            run_id,
        )

        type_issues = (
            find_type_conversion_issues(
                raw_df,
                staging_df,
                run_id,
            )
        )

        # ====================================================
        # 5. DUPLICATE REMOVAL
        # ====================================================

        staging_df, duplicate_issues = (
            remove_staging_duplicates(
                staging_df,
                run_id,
            )
        )

        # ====================================================
        # 6. LOAD STAGING
        # ====================================================

        load_staging(
            engine,
            staging_df,
        )

        # ====================================================
        # 7. WAREHOUSE CLEANING
        # ====================================================

        warehouse_df, warehouse_issues = (
            warehouse_clean(
                staging_df,
                run_id,
            )
        )

        # ====================================================
        # 8. FORMULA / RELATIONSHIP CHECKS
        # ====================================================

        # warehouse_df = calculate_formula_checks(
        #     warehouse_df
        # )

        # ====================================================
        # 9. LOAD DIMENSIONS
        # ====================================================

        load_dimensions(
            engine,
            warehouse_df,
        )

        # ====================================================
        # 10. LOAD FACT TABLE
        # ====================================================

        load_fact_table(
            engine,
            warehouse_df,
            run_id,
        )

        # ====================================================
        # 11. LOAD QUALITY ISSUES
        # ====================================================

        load_quality_issues(
            engine,
            [
                type_issues,
                duplicate_issues,
                warehouse_issues,
            ],
        )

        # ====================================================
        # 12. RUN STATISTICS
        # ====================================================

        duplicate_count = len(
            duplicate_issues
        )

        rejected_count = (
            len(warehouse_issues)
            if not warehouse_issues.empty
            else 0
        )

        update_run(
            engine,
            run_id,
            "SUCCESS",
            raw_row_count=raw_count,
            staging_row_count=len(
                staging_df
            ),
            warehouse_row_count=len(
                warehouse_df
            ),
            duplicate_count=duplicate_count,
            rejected_count=rejected_count,
        )

        logger.info(
            "ETL run %s completed successfully.",
            run_id,
        )

    except Exception as exc:

        logger.exception(
            "ETL pipeline failed."
        )

        update_run(
            engine,
            run_id,
            "FAILED",
            error_message=str(exc),
        )

        raise

In [ ]:
run_pipeline()

2026-10-02 16:43:08,184 | INFO | Creating PostgreSQL database objects.
2026-10-02 16:43:08,217 | INFO | Started ETL run: 28fc4858-a190-43e6-8af6-7749ee4953f7
2026-10-02 16:43:08,217 | INFO | Reading source file: c:\Users\msii\Desktop\work\interview\armani\armani_prj\Sample_Data.xlsx
2026-10-02 16:43:08,681 | INFO | Source contains 7839 rows.
2026-10-02 16:43:10,323 | INFO | Loaded 7839 rows into raw.source_sales.
2026-10-02 16:43:11,182 | INFO | Removed 39 Product + Week duplicates.


In [113]:
%pip install psycopg2

     ---------------------------------------- 2.8/2.8 MB 3.6 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
